# 3b. RAG Pipeline with LangChain

This notebook reimplements the same RAG pipeline from Notebook 03 using **LangChain** instead of LlamaIndex, demonstrating proficiency with both frameworks. It uses the same ChromaDB index, embedding model, and GPT-4o-mini with Chain-of-Thought prompting.

**Components:**
- `langchain_openai.ChatOpenAI` for LLM
- `langchain_huggingface.HuggingFaceEmbeddings` for embeddings
- `langchain_chroma.Chroma` for vector store
- `ChatPromptTemplate` for structured prompts
- Manual retrieval + reranking + generation pipeline

**Scope:** Runs on a 50-query sample to demonstrate functionality and compare results with the LlamaIndex implementation.

In [1]:
import json
import os
import re
from pathlib import Path
from collections import Counter

import pandas as pd
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from sentence_transformers import CrossEncoder

In [2]:
load_dotenv(Path("..") / ".env")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
CHROMA_DIR = BASE_DIR / "chroma_db"
RESULTS_DIR = BASE_DIR / "results"

SAMPLE_SIZE = 50  # Subset for demonstration

## 3b.1 Load Models and Vector Store

In [3]:
# Embedding model (same as LlamaIndex pipeline)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

# LLM (same config as v3: temp=0 deterministic, max_tokens=300 for CoT)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

# Load existing ChromaDB (created in Notebook 02 with LlamaIndex)
vectorstore = Chroma(
    collection_name="pubmed_rag",
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
)

# Retriever: top-20 by embedding similarity
retriever = vectorstore.as_retriever(search_kwargs={"k": 20})

# Cross-encoder reranker (same model as LlamaIndex pipeline)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print(f"LangChain pipeline configured")
print(f"  Vector store: {vectorstore._collection.count()} chunks")
print(f"  LLM: gpt-4o-mini (temp=0, max_tokens=300)")
print(f"  Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LangChain pipeline configured
  Vector store: 43806 chunks
  LLM: gpt-4o-mini (temp=0, max_tokens=300)
  Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


## 3b.2 Load Queries

In [4]:
queries = []
with open(DATA_DIR / "pubmedqa.jsonl", "r") as f:
    for line in f:
        queries.append(json.loads(line))

# Use a sample for demonstration
sample_queries = queries[:SAMPLE_SIZE]
print(f"Loaded {len(queries)} total queries, using {len(sample_queries)} for demo")

Loaded 500 total queries, using 50 for demo


## 3b.3 Define Prompt and Pipeline Functions

In [5]:
SYSTEM_PROMPT = """You are a medical expert answering yes/no/maybe questions based on PubMed abstracts.
Rules:
- First, reason step-by-step about what the evidence says (2-3 sentences).
- Then give your final answer on a new line in the format: "Final Answer: yes" or "Final Answer: no" or "Final Answer: maybe"
- Answer "yes" if the evidence supports the claim.
- Answer "no" if the evidence contradicts or does not support the claim.
- Answer "maybe" ONLY if the evidence is genuinely mixed or insufficient to decide.
- Most questions have a definitive answer. Prefer "yes" or "no" over "maybe"."""

FEW_SHOT_EXAMPLES = """Example 1:
Question: Is increased gravitational stress beneficial for bone density?
Context: Studies show weight-bearing exercise increases bone mineral density by 2-8% in postmenopausal women...
Reasoning: The evidence indicates that weight-bearing exercise, which increases gravitational stress on bones, leads to measurable increases in bone mineral density (2-8%). This supports the claim that gravitational stress benefits bone density.
Final Answer: yes

Example 2:
Question: Does smoking cessation reduce cardiovascular risk?
Context: After 1 year of cessation, coronary heart disease risk drops by 50%. After 15 years, risk equals that of a non-smoker...
Reasoning: The evidence clearly shows that quitting smoking leads to substantial cardiovascular risk reduction - 50% within one year and full normalization after 15 years. This strongly supports the claim.
Final Answer: yes

Example 3:
Question: Is homeopathy effective for treating asthma?
Context: A systematic review of 6 RCTs found no significant difference between homeopathic treatments and placebo in lung function or symptom scores...
Reasoning: A systematic review of 6 randomized controlled trials found no significant difference between homeopathy and placebo. This is strong evidence against effectiveness, as systematic reviews of RCTs are high-quality evidence.
Final Answer: no

Example 4:
Question: Can MRI replace biopsy for diagnosing prostate cancer?
Context: MRI showed sensitivity of 91% but specificity of only 37%. While useful for risk stratification, results were inconsistent across centers...
Reasoning: While MRI has high sensitivity (91%), its low specificity (37%) means many false positives. Additionally, results are inconsistent across centers, making it unreliable as a biopsy replacement. It may complement but cannot replace biopsy.
Final Answer: maybe

"""

# LangChain ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{user_prompt}"),
])


def build_user_prompt(query_text, contexts):
    """Build the user prompt from query and retrieved contexts."""
    context_parts = [f"[Document {i+1}]\n{ctx}" for i, ctx in enumerate(contexts)]
    context = "\n\n".join(context_parts)
    return f"""{FEW_SHOT_EXAMPLES}Now answer this question:

Context:
{context}

Question: {query_text}

Reasoning:"""


def extract_answer(text):
    """Extract yes/no/maybe from LLM response."""
    text_clean = text.strip().lower()
    match = re.search(r'final answer:\s*(yes|no|maybe)\b', text_clean, re.I)
    if match:
        return match.group(1).lower()
    if text_clean in ("yes", "no", "maybe"):
        return text_clean
    match = re.search(r'\b(yes|no|maybe)\b', text_clean, re.I)
    return match.group(1).lower() if match else "unknown"


def majority_vote(answers):
    """Return the majority answer from a list of answers."""
    valid = [a for a in answers if a in ("yes", "no", "maybe")]
    if not valid:
        return "unknown"
    return Counter(valid).most_common(1)[0][0]


def rerank_docs(query, docs, top_n=5):
    """Rerank documents using cross-encoder."""
    if not docs:
        return []
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:top_n]]


print("Pipeline functions defined")

Pipeline functions defined


## 3b.4 Run Pipeline

In [6]:
results_lc = []

for i, q in enumerate(sample_queries):
    # Step 1: Retrieve (LangChain retriever)
    docs_retrieved = retriever.invoke(q["query"])
    
    # Step 2: Rerank (cross-encoder)
    docs_reranked = rerank_docs(q["query"], docs_retrieved, top_n=5)
    contexts = [doc.page_content for doc in docs_reranked]
    
    # Step 3: Generate (single deterministic call, temperature=0)
    user_prompt = build_user_prompt(q["query"], contexts)
    chain = prompt_template | llm
    response = chain.invoke({"user_prompt": user_prompt})
    raw = response.content
    final_answer = extract_answer(raw)
    
    results_lc.append({
        "id": q["id"],
        "query": q["query"],
        "ground_truth": q["ground_truth"],
        "generated_answer": final_answer,
        "num_docs": len(docs_reranked),
    })
    
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(sample_queries)}...")

print(f"\nDone! Processed {len(results_lc)} queries ({len(results_lc)} API calls)")

  Processed 10/50...


  Processed 20/50...


  Processed 30/50...


  Processed 40/50...


  Processed 50/50...

Done! Processed 50 queries (50 API calls)


## 3b.5 Results and Comparison

In [7]:
# LangChain results
df_lc = pd.DataFrame(results_lc)
df_lc["correct"] = df_lc.apply(
    lambda r: r["generated_answer"] == r["ground_truth"].lower(), axis=1
)
lc_accuracy = df_lc["correct"].mean()

print("=" * 60)
print("  LangChain Pipeline Results")
print("=" * 60)
print(f"\nSample size: {len(df_lc)} queries")
print(f"Accuracy: {lc_accuracy:.2%}")
print(f"\nPrediction distribution:")
print(df_lc["generated_answer"].value_counts().to_string())

# Compare with LlamaIndex results (same sample)
df_llamaindex = pd.read_csv(RESULTS_DIR / "rag_summary.csv")
df_li_sample = df_llamaindex.head(SAMPLE_SIZE)
li_accuracy = df_li_sample["correct"].mean()

print(f"\n{'=' * 60}")
print(f"  Comparison: LangChain vs LlamaIndex (first {SAMPLE_SIZE} queries)")
print(f"{'=' * 60}")
print(f"\n{'Framework':<15} {'Accuracy':>10}")
print(f"{'─' * 25}")
print(f"{'LlamaIndex':<15} {li_accuracy:>10.2%}")
print(f"{'LangChain':<15} {lc_accuracy:>10.2%}")
print(f"{'─' * 25}")
print(f"{'Difference':<15} {abs(lc_accuracy - li_accuracy):>10.2%}")
print(f"\nBoth frameworks produce comparable results using the same")
print(f"models, embeddings, and prompts.")

  LangChain Pipeline Results

Sample size: 50 queries
Accuracy: 54.00%

Prediction distribution:
generated_answer
yes      27
maybe    18
no        5

  Comparison: LangChain vs LlamaIndex (first 50 queries)

Framework         Accuracy
─────────────────────────
LlamaIndex          58.00%
LangChain           54.00%
─────────────────────────
Difference           4.00%

Both frameworks produce comparable results using the same
models, embeddings, and prompts.


In [8]:
# Save LangChain results
df_lc.to_csv(RESULTS_DIR / "langchain_results.csv", index=False)
print(f"Saved LangChain results to results/langchain_results.csv")

# Show some examples
print(f"\n=== Sample Results ===")
for _, row in df_lc.head(5).iterrows():
    status = "correct" if row["correct"] else "wrong"
    print(f"  [{status}] Q: {row['query'][:80]}...")
    print(f"         GT: {row['ground_truth']}, Pred: {row['generated_answer']}")
    print()

Saved LangChain results to results/langchain_results.csv

=== Sample Results ===
  [correct] Q: Is anorectal endosonography valuable in dyschesia?...
         GT: yes, Pred: yes

  [correct] Q: Is there a connection between sublingual varices and hypertension?...
         GT: yes, Pred: yes

  [wrong] Q: Is the affinity column-mediated immunoassay method suitable as an alternative to...
         GT: yes, Pred: maybe

  [correct] Q: Does a physician's specialty influence the recording of medication history in pa...
         GT: yes, Pred: yes

  [correct] Q: Locoregional opening of the rodent blood-brain barrier for paclitaxel using Nd:Y...
         GT: yes, Pred: yes



## Summary

This notebook demonstrates that the RAG pipeline can be implemented with **LangChain** using the same components:

| Component | LlamaIndex (Notebook 03) | LangChain (This notebook) |
|---|---|---|
| **LLM** | `llama_index.llms.openai.OpenAI` | `langchain_openai.ChatOpenAI` |
| **Embeddings** | `llama_index.embeddings.huggingface.HuggingFaceEmbedding` | `langchain_huggingface.HuggingFaceEmbeddings` |
| **Vector Store** | `llama_index.vector_stores.chroma.ChromaVectorStore` | `langchain_chroma.Chroma` |
| **Prompt** | `ChatMessage` (manual) | `ChatPromptTemplate` |
| **Reranker** | `SentenceTransformerRerank` | `CrossEncoder` (direct) |

Both implementations achieve comparable accuracy, confirming that the pipeline's performance is driven by the choice of models and prompts, not the orchestration framework.